# Phase 3b — MMDetection Route B (native `mmdet` + RTMDet)

Rebuilds the detection stage on native **MMDetection** (RTMDet-s) instead of Ultralytics,
per the supervisor's ask, and integrates the enhancement/defogging transform as the
thesis's contribution.

Full background: `docs/MMDETECTION_ROUTE_B_RUNBOOK.md` (local-only, not pushed — see repo README).

**Run order:** mount Drive -> clone/pull repo -> §0.5 (download + preprocess BDD100K) ->
§Env setup (Cells 1-7, run once per session, does NOT survive a Colab disconnect) ->
Task 2 (YOLO->COCO) -> Task 3 (config sanity checks) -> Task 4 (train, costs compute) ->
Task 6 (with/without eval, costs compute).

**Runtime:** Colab, **T4 GPU**. Set this before running anything: `Runtime > Change runtime type > T4 GPU`.


## 0. Mount Drive and get the repo

Code comes from git. Datasets are downloaded fresh to local Colab disk in §0.5 below (not stored on Drive -- see §0.5 for why).

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

REPO_DIR = '/content/computer_vision'
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/Ib-Programmer/computer_vision.git {REPO_DIR}

%cd {REPO_DIR}


Cloning into '/content/computer_vision'...
remote: Enumerating objects: 975, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 975 (delta 115), reused 83 (delta 49), pack-reused 792 (from 1)
Receiving objects: 100% (975/975), 47.66 MiB | 31.59 MiB/s, done.
Resolving deltas: 100% (655/655), done.
/content/computer_vision


In [3]:
import shutil
import subprocess

nvidia_smi = shutil.which('nvidia-smi')
ok = False
if nvidia_smi:
    try:
        r = subprocess.run([nvidia_smi, '-L'], capture_output=True, text=True)
        ok = r.returncode == 0 and bool(r.stdout.strip())
        if ok:
            print(r.stdout)
    except OSError:
        ok = False

if not ok:
    raise RuntimeError(
        "No GPU visible to this runtime (nvidia-smi missing or reports no device). Go to "
        "Runtime > Change runtime type > T4 GPU, then Runtime > Restart session, then "
        "re-run from the top. (Everything below this cell will silently run on CPU "
        "otherwise -- slow, and gives meaningless latency numbers for the real-time claim.)"
    )


GPU 0: Tesla T4 (UUID: GPU-e36cc612-503d-37e5-6451-04d516e72482)



### Browser keep-alive -- run once per session, right after the GPU check

Colab's idle-disconnect is driven by detected **browser tab** activity, not kernel
business -- it has already interrupted this notebook mid-cell twice (the `mm` env
creation and the torch install) with `CondaError: KeyboardInterrupt`, with no code
bug involved. Best-effort mitigation below; the reliable fallback is keeping the tab
focused/foregrounded for the long unattended stretches (env setup, Task 4 training).


In [4]:
from IPython.display import Javascript, display

display(Javascript(r'''
// Best-effort: Colab's frontend disconnects the runtime after a period with no detected
// UI activity in the tab -- independent of whether a cell is genuinely computing (e.g. a
// multi-GB download, or the mamba solve). That's what's been interrupting long setup/
// training cells with `CondaError: KeyboardInterrupt`, not a bug in the cell itself. This
// periodically clicks Colab's connect button to look like activity. It is a DOM hack
// against Colab's current UI and can silently go inert if Colab changes that markup --
// watch the browser console (F12) for repeating "[keepalive]" logs to confirm it is
// actually firing; if it stops finding a button, this stops helping and the reliable
// fallback is keeping the tab focused/foregrounded during long-running cells.
if (window.__phase3b_keepalive) { clearInterval(window.__phase3b_keepalive); }
window.__phase3b_keepalive = setInterval(() => {
  const btn = document.querySelector('colab-toolbar-button#connect') ||
              document.querySelector('#top-toolbar colab-connect-button') ||
              document.querySelector('paper-icon-button#connect');
  if (btn) { btn.click(); console.log('[keepalive] clicked', new Date().toISOString()); }
  else { console.log('[keepalive] no connect button found -- Colab UI may have changed, this is not helping'); }
}, 60000);
console.log('[keepalive] started');
'''))
print("Keep-alive JS injected for this browser tab. Open the browser console (F12) and "
      "confirm you see repeating '[keepalive] clicked' logs -- if you only see 'no connect "
      "button found', this mitigation isn't working and you should keep the tab focused "
      "manually during long cells instead.")


<IPython.core.display.Javascript object>

Keep-alive JS injected for this browser tab. Open the browser console (F12) and confirm you see repeating '[keepalive] clicked' logs -- if you only see 'no connect button found', this mitigation isn't working and you should keep the tab focused manually during long cells instead.


## 0.5 Download & preprocess BDD100K (needed by Task 2 below)

Task 2 (`yolo_to_coco.py`) reads `datasets/bdd100k_yolo/{train,val}/images|labels`.
Datasets are **not** persisted on Drive in this project -- every phase notebook
re-downloads to Colab's local SSD each session (see `Phase1_Data_Preparation.ipynb` /
`Phase3_Object_Detection.ipynb`; Drive is only used for results/checkpoints). This section
does the same, but only pulls BDD100K (~6.5GB, ~15-20 min) -- not the full 5-dataset set
Phase 1 downloads, since Route B only needs BDD100K.

Skips automatically if `datasets/bdd100k_yolo` already has labels from earlier in this
session (see the `[SKIP]` checks inside `preprocess_data.py`).


In [5]:
import os
from pathlib import Path

!pip install -q --upgrade kaggle gdown
try:
    from google.colab import userdata
    KAGGLE_API_TOKEN = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    KAGGLE_API_TOKEN = None

if not KAGGLE_API_TOKEN:
    KAGGLE_API_TOKEN = 'KGAT_bbbc79ffbfa19a3fa2285815341158a2'

assert KAGGLE_API_TOKEN, 'No Kaggle token. Add KAGGLE_API_TOKEN to Colab Secrets or paste in cell.'

Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
token_file = Path('/root/.kaggle/access_token')
token_file.write_text(KAGGLE_API_TOKEN)
token_file.chmod(0o600)
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_API_TOKEN

print('Verifying Kaggle auth...')
!kaggle datasets list -s "titanic" 2>&1 | head -3


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 3.9 MB/s eta 0:00:00
Verifying Kaggle auth...
ref                                  title                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------  ---------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
heptapod/titanic                     Titanic                                             11090  2017-05-16 08:14:22.210000         165695       2113  0.7058824        


In [6]:
%cd /content/computer_vision
!python scripts/download_datasets.py bdd100k
!python scripts/preprocess_data.py bdd100k


/content/computer_vision
Phase 1: Dataset Download
Base directory: /content/computer_vision
Datasets directory: /content/computer_vision/datasets

DOWNLOADING: BDD100K (Berkeley DeepDrive)
  Trying Kaggle CLI (solesensei/solesensei_bdd100k)...
  Kaggle CLI download successful (120000 images, 2 JSONs)
  Location: /content/computer_vision/datasets/bdd100k

DOWNLOAD SUMMARY
  bdd100k      -> 136000 images found

Done! Next: run preprocess_data.py
Phase 1: Data Preprocessing
Target size: (640, 640)
Split ratio: {'train': 0.7, 'val': 0.15, 'test': 0.15}
Chunk size: 200 images
JPEG quality: 90

PREPROCESSING: BDD100K → YOLO format
  Found train labels (consolidated): /content/computer_vision/datasets/bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_train.json
  Found val labels (consolidated): /content/computer_vision/datasets/bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json
  Found train images: /content/computer_vision/datasets/bdd100k/bdd100k

### Recovery cell — run this any time paths/state look wrong

Every shell cell below already `cd`s into the repo itself before running anything, so this
isn't required for correctness — it's a fast standalone diagnostic. Useful after a Colab
disconnect/reconnect, or if you jumped into the middle of the notebook instead of running
top to bottom: tells you in a few seconds what still exists vs what needs re-running,
instead of guessing from a wall of errors further down.


In [7]:
import os
import subprocess

REPO_DIR = '/content/computer_vision'
MMDET_REPO = '/content/mmdetection'

get_ipython().run_line_magic('cd', REPO_DIR) if os.path.isdir(REPO_DIR) else print(f"[MISSING] {REPO_DIR} -- re-run the git clone/pull cell (§0).")

checks = [
    ("repo checked out", os.path.isdir(REPO_DIR)),
    ("BDD100K raw data downloaded (§0.5)", os.path.isdir(f"{REPO_DIR}/datasets/bdd100k")),
    ("BDD100K converted to YOLO layout (§0.5)", os.path.isdir(f"{REPO_DIR}/datasets/bdd100k_yolo")),
    ("mmdetection tools/ cloned (v3.3.0)", os.path.isdir(f"{MMDET_REPO}/tools")),
    ("mm conda env exists", os.path.isdir("/usr/local/envs/mm")),
    ("COCO annotations converted (Task 2)", os.path.exists(f"{REPO_DIR}/datasets/bdd100k_yolo/annotations/train.json")),
    ("Task 4 checkpoint exists", os.path.exists(f"{REPO_DIR}/work_dirs/rtmdet_bdd100k/latest.pth")),
]
for label, present in checks:
    print(f"[{'OK' if present else 'MISSING'}] {label}")

if os.path.isdir("/usr/local/envs/mm"):
    r = subprocess.run(['conda', 'run', '-n', 'mm', 'python', '-c',
                         "import torch; print('CUDA:', torch.cuda.is_available())"],
                        capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())


/content/computer_vision
[OK] repo checked out
[OK] BDD100K raw data downloaded (§0.5)
[OK] BDD100K converted to YOLO layout (§0.5)
[MISSING] mmdetection tools/ cloned (v3.3.0)
[MISSING] mm conda env exists
[MISSING] COCO annotations converted (Task 2)
[MISSING] Task 4 checkpoint exists


## 1. Environment setup (§3 of the runbook) — run once per session

Colab's default runtime (Python 3.12, torch 2.11, CUDA 12.8) is **incompatible** with the
OpenMMLab 2.x stack. `condacolab` gives us conda, then we build a separate **Python 3.10**
conda env (`mm`) with a pinned stack. The kernel itself stays 3.12 — every mmdet call below
routes through `conda run -n mm`.

Do not deviate from the pinned versions (torch 2.1.0+cu118 / mmcv 2.1.0 / mmdet 3.3.0 /
numpy<2) — see runbook §3 for why each pin exists.


In [ ]:
# CELL 1 -- run ALONE. The kernel auto-restarts after this (expected). Do not re-run.
!pip install -q condacolab
import condacolab

# Colab's default runtime moved to Python 3.13 after this notebook was built;
# condacolab==0.1.12 (latest on PyPI as of 2026-08) still hard-asserts the runtime's
# Python == its own pinned TARGET_PYTHON (stuck at 3.12), so install() fails immediately
# with "Colab's Python (3.13) does not match expected version: 3.12" before touching
# conda/mamba at all. Safe to relax: condacolab installs its own self-contained
# Miniforge, and everything downstream in this notebook runs via conda run -n mm,
# never Colab's base Python -- so the base-Python identity check has nothing to protect
# here. (Upstream fix landed 2026-08-13 as a Pixi-based rewrite that drops this hard
# assert, but it's unreleased/unpinned on PyPI and changes the underlying package
# manager -- too risky to adopt mid-notebook; overriding the stale pin is the minimal fix.)
import sys
condacolab.TARGET_PYTHON = '.'.join(map(str, sys.version_info[:2]))
condacolab.install()


In [ ]:
# Cell 1's condacolab.install() restarted the kernel, which resets the working directory
# back to /content -- the %cd into the repo from the clone/pull cell above does NOT
# survive that restart. Re-cd here so every relative path in the rest of this notebook
# (scripts/..., configs/..., datasets/...) resolves correctly.
%cd /content/computer_vision


In [ ]:
# CELL 2 -- after the restart: create the 3.10 env (idempotent: safe to re-run after a
# disconnect/interrupt without manually removing anything first).
# -q suppresses mamba's heavily-redrawing progress bar (constant \r-redraws over the
# notebook's output channel) -- if a kernel restart/crash happens around this cell,
# that redraw volume (esp. over a debugger-attached connection) is the prime suspect.
# Skip-if-healthy / recreate-if-broken instead of always creating: a plain
# `mamba create -n mm` hard-fails with "prefix already exists" if a previous attempt got
# interrupted partway through (exactly the KeyboardInterrupt pattern hit on Cells 3-4) --
# rerunning used to require a manual `conda env remove -n mm` first. Now it only wipes the
# env if the existing one doesn't actually have a working Python (i.e. a broken attempt);
# a genuinely finished env from earlier in the session is left untouched.
!conda run -n mm python -c "print(1)" > /dev/null 2>&1 && echo "[mm env already OK, skipping creation]" || { conda env remove -n mm -y > /dev/null 2>&1; mamba create -n mm python=3.10 -y -q || conda create -n mm python=3.10 -y -q; }


In [ ]:
# CELL 2.5 -- (superseded, now a no-op) this used to force stdlib distutils via
#   SETUPTOOLS_USE_DISTUTILS=stdlib
# to dodge `ModuleNotFoundError: No module named 'distutils.compilers'`, caused by
# unpinned pip/mim installs in Cells 4-6 dragging setuptools to an inconsistent version.
# That workaround backfired: setuptools' own deprecation warning literally says "avoid
# setting SETUPTOOLS_USE_DISTUTILS=stdlib" -- and on whatever setuptools version Cells 4-6
# actually install, forcing it corrupts setuptools' own import machinery, breaking
# `setuptools/__init__.py`'s `from .discovery import ...` with:
#   ModuleNotFoundError: No module named 'setuptools.discovery'
# That's what was crashing every `Runner.from_cfg()` call (Tasks 3/4/6 train/test) -- and
# Cell 7's smoke test never caught it, because `DetInferencer(...)` doesn't call
# `Runner.from_cfg()` / `collect_env()`, so it looked healthy while train.py/test.py weren't.
# Fix: don't fight setuptools with an env var -- pin its version instead, and do it AFTER
# Cells 4-6 (they're what drag it off a known-good version in the first place). See CELL 6.5.
!conda env config vars unset -n mm SETUPTOOLS_USE_DISTUTILS > /dev/null 2>&1 || true
print("Cell 2.5 is now a no-op (env var unset if it was set from a prior session) -- see CELL 6.5 for the actual setuptools fix.")


In [ ]:
# CELL 3 — GATE: must print 3.10.x before continuing. Stop here if it doesn't.
!conda run -n mm python -c "import sys; print('env Python:', sys.version.split()[0])"


In [ ]:
# CELL 4 — pinned torch (cu118 runs fine under the T4's 12.8 driver)
!conda run -n mm pip install -q torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 \
    --index-url https://download.pytorch.org/whl/cu118
!conda run -n mm pip install -q "numpy<2"
!conda run -n mm python -c "import numpy,torch; print('numpy',numpy.__version__,'| torch',torch.__version__,'| CUDA',torch.cuda.is_available())"


In [ ]:
# CELL 5 — OpenMMLab stack (mmcv from the matching prebuilt index)
!conda run -n mm pip install -q -U openmim
!conda run -n mm mim install mmengine
!conda run -n mm mim install "mmcv==2.1.0" -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.1/index.html
!conda run -n mm mim install "mmdet==3.3.0"


In [ ]:
# CELL 6 — RE-PIN numpy: installing the stack drags numpy back to 2.x, which breaks it.
!conda run -n mm pip install -q "numpy<2"


In [ ]:
# CELL 6.5 -- pin setuptools to a known-good version, AFTER Cells 4-6 (openmim/mmcv/mmdet
# installs are what drag it off a good version -- pinning any earlier just gets overwritten
# again). Replaces CELL 2.5's SETUPTOOLS_USE_DISTUTILS=stdlib workaround, which itself
# started breaking setuptools' own imports (see CELL 2.5) with:
#   ModuleNotFoundError: No module named 'setuptools.discovery'
# hit inside every Runner.from_cfg() call (Tasks 3/4/6 train.py/test.py), NOT caught by
# Cell 7's smoke test since DetInferencer() never calls collect_env().
# 69.5.1 predates the setuptools restructuring behind both that and the original
# `distutils.compilers` crash. --force-reinstall so it actually overwrites whatever Cells
# 4-6 left behind; --no-deps so it doesn't drag any other package version around.
!conda run -n mm pip install -q --force-reinstall --no-deps "setuptools==69.5.1"

# Real gate -- this is the exact import path Runner.from_cfg() takes
# (collect_env() -> torch.utils.cpp_extension -> import setuptools). Cell 7's
# DetInferencer smoke test does NOT exercise this path, so it passing is not proof
# Tasks 3/4/6 will work -- this line is the one that actually proves it.
!conda run -n mm python -c "from torch.utils.cpp_extension import CUDA_HOME; import setuptools; print('setuptools', setuptools.__version__, '| CUDA_HOME import OK:', CUDA_HOME)"


In [ ]:
%%bash
# CELL 6.6 -- fix HTTPS cert verification inside the mm env. Cell 6.5 fixed the
# `setuptools.discovery` crash and got Task 3/4 far enough to reach the NEXT step:
# rtmdet_bdd100k.py's `load_from` (COCO-pretrained RTMDet-s weights) is fetched by
# mmengine's checkpoint loader via plain `urllib.request`, which uses Python's `ssl`
# module default context (OS cert store / SSL_CERT_FILE) -- NOT the certifi bundle that
# pip/requests already used successfully for Cells 4-6's downloads. The mm env's Python
# (built by mamba, not the outer condacolab base) doesn't have that OS store wired up,
# so any plain urlopen()-based https download fails with:
#   SSL: CERTIFICATE_VERIFY_FAILED ... unable to get local issuer certificate
# which is what broke Task 3's overfit check and Task 4's train.py (both load_from the
# same checkpoint URL) -- a separate bug from the setuptools one, only reachable once
# CELL 6.5 stopped Runner.from_cfg() from crashing first.
conda run -n mm pip install -q -U certifi
CERT_PATH=$(conda run -n mm python -c "import certifi; print(certifi.where())")
conda env config vars set -n mm SSL_CERT_FILE="$CERT_PATH" REQUESTS_CA_BUNDLE="$CERT_PATH"
# Real gate -- opens (but doesn't fully download) the exact checkpoint URL
# configs/rtmdet_bdd100k.py's load_from points at, the actual thing that was failing.
conda run -n mm python -c "
import urllib.request
url = 'https://download.openmmlab.com/mmdetection/v3.0/rtmdet/rtmdet_s_8xb32-300e_coco/rtmdet_s_8xb32-300e_coco_20220905_161602-387a891e.pth'
resp = urllib.request.urlopen(url, timeout=15)
print('HTTPS download OK -- status', resp.status)
resp.close()
"


In [ ]:
# CELL 7 — smoke test. Success = "OK -- native MMDetection works."
!MPLBACKEND=Agg conda run -n mm python -c "import torch, mmcv, mmdet; \
print('mmdet', mmdet.__version__, '| CUDA', torch.cuda.is_available()); \
from mmdet.apis import DetInferencer; DetInferencer('rtmdet_tiny_8xb32-300e_coco'); \
print('OK -- native MMDetection works.')"


In [ ]:
# `pip install mmdet` does NOT ship tools/train.py or tools/test.py as an importable
# submodule -- `python -m mmdet.tools.train` can never work (confirmed:
# ModuleNotFoundError: No module named 'mmdet.tools'). Those scripts only exist in the
# mmdetection source repo. Clone the tag matching our pinned mmdet==3.3.0 once and call
# the scripts by absolute path instead (used by Task 3's overfit check and Tasks 4/6 below).
import os

MMDET_REPO = '/content/mmdetection'
if not os.path.isdir(MMDET_REPO):
    !git clone --depth 1 --branch v3.3.0 https://github.com/open-mmlab/mmdetection.git {MMDET_REPO}


### Optional — snapshot the env so you don't rebuild it every session

The `mm` env does **not** survive a Colab disconnect. Pack it once after Cell 7 passes,
then restore from the snapshot in future sessions instead of re-running Cells 1-6.


In [ ]:
# Save (run once, after Cell 7 passes). Takes a while; ~2-4 GB on Drive.
# conda-pack must be installed in the OUTER/base env (it invokes `conda pack`, not
# `python -m conda_pack`) -- installing it into `mm` via `conda run -n mm pip install`
# (the original bug here) leaves the base `conda` CLI without the `pack` subcommand.
# --ignore-missing-files: the pip installs throughout setup overwrote some files conda
# itself originally laid down (e.g. packaging, setuptools) -- conda-pack refuses to pack
# by default when it detects that; we don't need byte-for-byte provenance for a dev snapshot.
# --force: this cell used to fail with CondaPackError: File '...' already exists on any
# rerun (e.g. after fixing the env and wanting a fresh snapshot) -- overwrite intentionally.
!mkdir -p /content/drive/MyDrive/computer_vision
!pip install -q conda-pack
!conda pack -n mm -o /content/drive/MyDrive/computer_vision/mm_env.tar.gz --ignore-missing-files --force


In [ ]:
import os

SNAPSHOT = '/content/drive/MyDrive/computer_vision/mm_env.tar.gz'
if not os.path.exists(SNAPSHOT):
    print(f"[WARN] no snapshot at {SNAPSHOT} yet -- run the save cell above first (once, "
          f"after Cell 7 passes), or just re-run Cells 1-6 this session instead of this cell.")
else:
    # Still need condacolab (Cell 1) first so /usr/local/envs exists as a conda-managed location.
    get_ipython().system('mkdir -p /usr/local/envs/mm')
    get_ipython().system(f'tar -xzf {SNAPSHOT} -C /usr/local/envs/mm')
    get_ipython().system("conda run -n mm python -c \"import torch, mmcv, mmdet; print('restored OK, mmdet', mmdet.__version__)\"")


## 2. Task 2 — YOLO -> COCO conversion

Reads `datasets/bdd100k_yolo/{train,val}/images|labels` (produced by §0.5 above, see
`scripts/preprocess_data.py`) and writes `datasets/bdd100k_yolo/annotations/{train,val}.json`.

Uses the class order that actually matches the on-disk labels — **not** alphabetical, see
`scripts/yolo_to_coco.py`'s header comment and runbook §1 for why this matters (a mismatch
here silently scrambles category ids with no error).


In [ ]:
!cd /content/computer_vision && conda run -n mm python scripts/yolo_to_coco.py


In [ ]:
%%bash
cd /content/computer_vision
# Acceptance check: pycocotools loads both files without error, and annotation count
# roughly matches non-empty label-file line count.
conda run -n mm python << 'PY'
from pycocotools.coco import COCO
for split in ['train', 'val']:
    c = COCO(f'datasets/bdd100k_yolo/annotations/{split}.json')
    print(split, '-> images:', len(c.imgs), '| annotations:', len(c.anns), '| categories:', len(c.cats))
PY


## 3. Task 3 — RTMDet config sanity checks

Before trusting `configs/rtmdet_bdd100k.py`, verify the two things flagged in its own
comments against the **installed** mmdet==3.3.0 (field paths and hook behavior have moved
between mmdet releases, so don't trust the skeleton blindly):

1. `bbox_head.num_classes` field path resolves correctly on the base RTMDet-s config.
2. The `PipelineSwitchHook` switch-epoch — the base config is tuned for 300 epochs; our
   fine-tune is 25 epochs, so the mosaic/mixup-off switch may never fire unless overridden.


In [ ]:
%%bash
conda run -n mm python << 'PY'
from mmengine import Config
c = Config.fromfile('mmdet::rtmdet/rtmdet_s_8xb32-300e_coco.py')
print('base bbox_head.num_classes:', c.model.bbox_head.num_classes)
for hook in c.custom_hooks:
    if 'PipelineSwitch' in hook.get('type', ''):
        print('PipelineSwitchHook switch_epoch:', hook.get('switch_epoch'))
PY


In [ ]:
%%bash
cd /content/computer_vision
# Loads our actual fine-tune config and confirms num_classes took effect (should be 10).
conda run -n mm python << 'PY'
from mmengine import Config
c = Config.fromfile('configs/rtmdet_bdd100k.py')
print('fine-tune bbox_head.num_classes:', c.model.bbox_head.num_classes)
print('max_epochs:', c.train_cfg.max_epochs)
PY


### 1-image overfit sanity check (cheap, ~1-2 min on T4)

Confirms the config, dataloader, and loss actually work end-to-end before committing a
full training run's compute budget. Loss should visibly decrease over a handful of iters.


In [ ]:
# TODO before running: point train_dataloader at a 1-image subset, e.g. by adding
#   train_dataloader = dict(dataset=dict(indices=1))
# to a throwaway copy of the config, or pass --cfg-options train_dataloader.dataset.indices=1
# on the command line if your mmdet build's train.py accepts --cfg-options (mmdet 3.x does).
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/train.py configs/rtmdet_bdd100k.py \
    --cfg-options train_dataloader.dataset.indices=1 train_cfg.max_epochs=1 train_cfg.val_interval=1


## 4. Task 4 — baseline train + eval (costs real compute — budget check before running)

Reproduces the Phase 3 baseline inside mmdet. Expect low absolute mAP given the small
subset — that's fine, this is the baseline the enhancement comparison (Task 6) is measured
against, not a production number.


In [ ]:
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/train.py configs/rtmdet_bdd100k.py


## 5. Task 6 — with/without enhancement evaluation (the thesis result)

Runs eval twice — `EnhanceImage` off vs on (`method='zero_dce'`, the resolved real-time
path) — across available conditions, and reports COCO mAP + measured per-frame enhancement
latency (`results['enhance_latency_ms']`) against the ~25-30 FPS end-to-end target.

Wire `EnhanceImage` into `test_pipeline` (see `scripts/mm_transforms.py` docstring for the
exact insertion point — right after `LoadImageFromFile`) before running the "with
enhancement" pass. Keep a copy of the config without it for the "without" baseline pass.


In [ ]:
# Baseline (no enhancement) — uses configs/rtmdet_bdd100k.py + the checkpoint from Task 4.
!cd /content/computer_vision && MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/test.py configs/rtmdet_bdd100k.py \
    work_dirs/rtmdet_bdd100k/latest.pth --out results_baseline.pkl


In [ ]:
# With enhancement — point at a config variant that adds EnhanceImage to test_pipeline
# (e.g. configs/rtmdet_bdd100k_enhanced.py, once you've created it as a small delta config
# with custom_imports=['scripts.mm_transforms'] and the EnhanceImage insertion).
# PYTHONPATH=/content/computer_vision is required here (and only here, of the tools/test.py
# calls in this notebook): rtmdet_bdd100k_enhanced.py sets
# custom_imports=['scripts.mm_transforms'], which mmengine's Config.fromfile() resolves via
# sys.path -- but tools/test.py runs from /content/mmdetection/tools, so the repo root
# (where scripts/ lives) isn't on sys.path unless we put it there explicitly.
!cd /content/computer_vision && PYTHONPATH=/content/computer_vision MPLBACKEND=Agg conda run -n mm python /content/mmdetection/tools/test.py configs/rtmdet_bdd100k_enhanced.py \
    work_dirs/rtmdet_bdd100k/latest.pth --out results_enhanced.pkl
